> **中文阅读版**：本文件是 `01_react_reasoning.ipynb` 的 Markdown 翻译副本。代码、输出、图表和路径保持原样；专有名词和标识符保留英文。
# LLM 提供商与 ReAct 循环

**Docker 镜像**：`ml4t`

**预测智能体（forecasting agent）** 通过选择要查询的内容、阅读返回结果，然后给出一个确定的数字来回答问题。这与前几章中的模型在程序形态上不同：那些模型接收准备好的特征矩阵，并据此生成预测。本 notebook 构建最小可用版本：一个在“思考自己还需要什么”与“采取行动获取它”之间交替的循环，并由一个无需改动循环即可替换的语言模型支撑。

**学习目标**：
- 通过同一个包含两个方法的接口调用五种语言模型后端中的任意一种，使同一份智能体代码既能对接免费的本地模型，也能对接商业 API
- 编写一个在推理与工具调用之间交替（即 **ReAct** 模式）的循环，并在产出预测或达到你设定的步数预算时停止
- 在依据模型回复采取行动之前，先按已声明的 action schema 校验该回复；若回复格式有误，则将其退回模型进行修正
- 读取已保存的运行记录，查看发起了哪些搜索、返回了什么，以及循环对每个结果做了什么处理

**书目参考**：第 24 章，第 24.1–24.2 节（从预测函数到智能体工作流；认知架构）

**前置条件**：无。这是第 24 章工作坊中的第一个 notebook。

In [2]:
"""LLM Providers and the ReAct Loop - multi-provider agent reasoning."""

import sys

from utils.paths import get_chapter_dir

sys.path.insert(0, str(get_chapter_dir(24)))

import json
from datetime import date

from agent_fixtures import get_live_question
from agent_observability import TRACES_DIR, RunTrace, trace_llm
from agent_pipeline import validate_probabilities
from agent_providers import ChatMessage, MockLLMClient, TokenUsage, create_llm_client
from agent_schemas import AgentForecastArtifact, AgentTrace, ForecastQuestion
from agent_tools import ToolExecutor, create_search_client, format_search_results

In [3]:
# RUN_LIVE=False (the default) replays a pinned claude-sonnet + Tavily run: the
# notebook reloads the saved trace named below, makes no API calls, and
# reproduces the captured ReAct session, so the outputs are stable and match the
# narrative. Set RUN_LIVE=True (with LLM_PROVIDER + LLM_API_KEY + TAVILY_API_KEY in .env) to
# fetch a fresh live question and run the agent against it; that path makes real
# calls and is not reproducible.
RUN_LIVE = False
PINNED_TRACE = "01_react_reasoning_20260615T191047Z_04eb6e7c603d.json"

LLM_PROVIDER = ""  # Blank reads .env; "mock" selects a synthetic fixture
MAX_STEPS = 6

## LLM 提供商协议

本章的每个 notebook 都通过同一个 `LLMClient` **协议**访问其语言模型：任何 provider 类都必须提供的一对方法签名，不共享基类，调用方也看不到任何厂商 SDK。

- `complete(messages) → str` 返回回复文本
- `complete_with_usage(messages) → (str, TokenUsage)` 返回文本以及 prompt 与 completion 的 token 计数，这正是使一次运行的成本可度量的原因

两个方法是有意设定的下限。provider 在这些方法之外提供的一切，例如流式输出、原生 tool calling 或思考预算，都会因厂商而异；而一旦智能体依赖其中任何一项，它便无法再被迁移。

下面的单元格发送一个算术问题。它的答案本身没有价值；它要确认的是：在循环开始为真实 prompt 花钱之前，客户端已构建、密钥已被接受、回复已经返回。在回放路径上没有客户端，因此回复来自已保存的记录。

**可选依赖**（用于接入真实 LLM 和网页搜索——不安装时本章以离线回放模式运行）：

```bash
uv pip install anthropic openai httpx        # cloud providers
# Ollama (free, local) - install from https://ollama.com, then:
#   ollama serve && ollama pull qwen3:8b
```

然后在仓库根目录的 `.env` 中设置你的 provider（先复制：`cp .env.example .env`）。你需要设置三项——**绝不要设置 base URL**：

- `LLM_PROVIDER` - `deepseek` | `openrouter` | `openai` | `anthropic` | `google` | `ollama` | `mock`
- `LLM_API_KEY` - 你为该 provider 准备的密钥
- `LLM_MODEL` - *可选*；留空则使用 provider 默认值
- `TAVILY_API_KEY` - 网页搜索（[tavily.com](https://tavily.com)）

实时模式会自动加载 `.env`。缺少实时凭证会抛出明确错误；仅在带标注的合成测试夹具（synthetic test fixture）中才选择 `mock`。运行 `uv run python check_env.py` 以确认将使用哪个模型。

In [4]:
if RUN_LIVE:
    llm = create_llm_client(LLM_PROVIDER)
    provider_name = llm.model_name
    response = llm.complete(
        [ChatMessage(role="user", content="What is 2 + 2? Reply with just the number.")]
    )
else:
    # Replay: reload the pinned trace; the provider name and warm-up response
    # come from the saved run, and no live client is created.
    pinned_run = RunTrace.load(TRACES_DIR / PINNED_TRACE)
    llm = None
    provider_name = pinned_run.provider
    response = pinned_run.params.get("warmup_response", "")

print(f"Mode:     {'LIVE' if RUN_LIVE else 'REPLAY (pinned trace)'}")
print(f"Provider: {provider_name}")
print(f"Response: {response[:200]}")

Mode:     REPLAY (pinned trace)
Provider: claude-sonnet-4-6
Response: 4


## 预测问题

智能体被指向一个**预测市场（prediction market）**问题：若某既定事件在既定日期前发生即偿付的合约，因此其价格就是市场认为该事件会发生的概率。Polymarket 公开发布这类合约，而一个尚未揭晓的问题在这里很有用，其原因在评估任何语言模型的预测能力时都至关重要。答案已存在于模型训练数据中的问题，考察的是记忆（recall），而非预测。未决问题根本不存在于训练数据中，因此要对其发表任何看法，唯一的办法就是去查。

在回放路径上，问题来自已保存的记录。在实时路径上，`get_live_question` 会调用 Polymarket API，当该调用失败时回退到一个固定的演示问题，因此 notebook 在无网络时仍可运行。

In [5]:
question = get_live_question() if RUN_LIVE else pinned_run.question_obj()
print(f"Question: {question.question}")
if question.resolution_date:
    print(f"Resolves: {question.resolution_date}")
if question.current_market_price is not None:
    print(f"Market:   {question.current_market_price:.0%}")
if question.resolved_outcome is not None:
    print(f"Outcome:  {'YES' if question.resolved_outcome == 1.0 else 'NO'}")
else:
    print("Status:   OPEN (unresolved)")

Question: Will no Fed rate cuts happen in 2026?
Resolves: 2026-12-31
Market:   71%
Status:   OPEN (unresolved)


## 构建 ReAct 智能体

ReAct 模式在一个循环中交替进行**推理**（思考要做什么）与**行动**（调用工具）。这里的智能体恰好有两种行动：

- `{"action": "search", "query": "..."}`：搜索网络以获取证据
- `{"action": "forecast", "p_yes": 0.XX, "rationale": "..."}`：给出一个概率

下面完整展示 prompt，而不是隐藏在辅助函数之后，因为模型被告知的内容与调用它的循环一样，都是这个程序的一部分。同样的两行动 schema 也被 AIA Forecaster（Alur 等，2025）使用，而后者正是本章其余部分要重建的系统。

In [6]:
SYSTEM_PROMPT = """\
You are a forecasting agent in a multi-agent forecasting system.

Your job:
1) Gather evidence by issuing web/news search queries when needed.
2) Then produce a binary probability forecast for the question.

You must follow the action schema exactly and output valid JSON only.
You must not browse prediction market prices unless they are explicitly provided."""
SYSTEM_PROMPT += (
    "\nCite a retained source URL or title in the rationale. Abstain if evidence is insufficient."
)

### 步骤 prompt

步骤 prompt 提供问题、可选的市场上下文以及 action schema。智能体每一步必须恰好输出一个 JSON 行动。

In [7]:
def build_step_prompt(q: ForecastQuestion, market_price: float | None = None) -> str:
    """Format the step prompt with question context."""
    prompt = f"QUESTION:\n{q.question}\n\n"
    if q.description:
        prompt += f"MARKET CONTEXT:\n{q.description}\n\n"
    if market_price is not None:
        prompt += f"MARKET IMPLIED PROBABILITY (p_yes):\n{market_price}\n\n"
    prompt += (
        "SEARCH BUDGET: search only when a specific fact is missing; do not "
        "reformulate a query you already ran. Once you have a base rate and the "
        "current signal, forecast.\n\n"
        "NEXT ACTION SCHEMA (output JSON only):\n"
        'If you need more info:\n{"action":"search","query":"..."}\n'
        "If you are ready to forecast:\n"
        '{"action":"forecast","p_yes":0.XX,"rationale":"short explanation '
        'grounded in evidence and base rates"}\n\n'
        'If evidence is insufficient: {"action":"abstain","rationale":"why"}\n'
        "Pick exactly one action."
    )
    return prompt

### 会话设置

一个会话以三部分状态开启。**截止日期（cutoff date）** 是搜索工具可以返回文档的最后日期，正是这一点防止智能体在预测过去的问题时读到答案。工具执行器持有搜索客户端，并且是该循环通往外部世界的唯一路径。消息列表从系统提示和第一步提示开始，每轮增加两个条目：模型说了什么，以及工具返回了什么。

In [8]:
def _init_react_session(
    question: ForecastQuestion,
    search_client,
    market_price: float | None,
) -> tuple[date | None, ToolExecutor, list[ChatMessage]]:
    """Build the cutoff date, tool executor, and initial messages for a ReAct run."""
    cutoff = date.fromisoformat(question.cutoff_date) if question.cutoff_date else None
    executor = ToolExecutor(search=search_client)
    messages = [
        ChatMessage(role="system", content=SYSTEM_PROMPT),
        ChatMessage(role="user", content=build_step_prompt(question, market_price)),
    ]
    return cutoff, executor, messages

### ReAct 循环

每次迭代：LLM 决定要做什么（搜索或预测），我们执行它，并把观察结果反馈回去。当智能体发出 `forecast` 动作或达到步数上限时，循环终止。`p_yes` 值必须是 [0, 1] 内的有限数，并附带一份指明所保留证据的理由说明。无效或不支持的输出会被拒绝。预算结束时可以弃权，而不是给出概率。

In [9]:
def run_react_agent(
    llm,
    search_client,
    question: ForecastQuestion,
    max_steps: int = 5,
    market_price: float | None = None,
) -> tuple[float | None, str, list[AgentTrace], TokenUsage]:
    if max_steps < 1:
        raise ValueError("max_steps must be positive")
    cutoff, executor, messages = _init_react_session(question, search_client, market_price)
    traces, sources = [], []
    total_tokens = TokenUsage()
    reason = "Budget exhausted without a supported forecast"
    for step in range(max_steps + 1):
        synthesis_only = step == max_steps
        if synthesis_only:
            if not sources:
                break
            messages.append(
                ChatMessage(
                    role="user",
                    content="No more searches. Cite retained evidence and forecast, or abstain.",
                )
            )
        try:
            raw, usage = llm.complete_with_usage(messages, json_mode=True)
        except Exception as exc:
            reason = f"Model call failed ({type(exc).__name__})"
            traces.append(AgentTrace(step=step, action="provider_error", llm_raw=reason))
            break
        total_tokens += usage
        messages.append(ChatMessage(role="assistant", content=raw))
        try:
            action = json.loads(raw)
            if not isinstance(action, dict):
                raise ValueError("Action must be an object")
        except (ValueError, TypeError):
            action = {}
        kind = action.get("action", "invalid")
        if kind == "search" and not synthesis_only:
            query = action.get("query", "")
            results = executor.execute_search(query, max_results=5, cutoff_date=cutoff)
            sources.extend(results)
            traces.append(
                AgentTrace(step=step, action="search", query=query, results=results, llm_raw=raw)
            )
            messages.append(ChatMessage(role="tool", content=format_search_results(results)))
            print(f"Step {step}: search -> {len(results)} eligible results")
            continue
        traces.append(AgentTrace(step=step, action=str(kind), llm_raw=raw))
        if kind == "abstain":
            reason = str(action.get("rationale") or "Insufficient evidence")
            break
        try:
            if kind != "forecast":
                raise ValueError("Invalid action or exhausted search budget")
            if isinstance(action["p_yes"], bool):
                raise ValueError("Probability must be a number, not a boolean")
            probability = float(action["p_yes"])
            validate_probabilities([probability])
            rationale = action.get("rationale")
            if not isinstance(rationale, str) or not any(
                (s.url and s.url in rationale) or (s.title and s.title in rationale)
                for s in sources
            ):
                raise ValueError("Forecast needs a rationale identifying retained evidence")
        except (KeyError, TypeError, ValueError) as exc:
            reason = str(exc)
            messages.append(ChatMessage(role="tool", content=reason))
            continue
        return probability, rationale, traces, total_tokens
    return None, reason, traces, total_tokens

## 运行智能体

智能体在每一轮做出选择：再次搜索，还是提交一个概率。下面记录的是会话，每轮一行。

在默认回放路径上，本次运行没有得出预测。模型不断搜索证据，以确认 2026 年利率路径是否已经确定，找到了正反两方各执一词的报道，并把每一轮都花在这上面。这是智能体在预算约束下的常见失败，也是 `run_react_agent` 返回一个显式的无答案值而不是概率的原因：调用方若把该值当作预测来读取，就会在智能体什么都没说的地方记录下一次自信的抛硬币。笔记本末尾的 mock 运行走另一条分支并提交结果。

In [10]:
if RUN_LIVE:
    search_client = create_search_client()
    search_name = type(search_client).__name__
    # Wrap the client so every prompt/response is captured for the run trace.
    tracer = trace_llm(llm, label="react_agent")
    print(f"Search: {search_name}\n")
    print(f"Question: {question.question}\n")
    p_yes, rationale, traces, tokens = run_react_agent(
        tracer, search_client, question, max_steps=MAX_STEPS
    )
else:
    # Replay: rehydrate the pinned agent and reconstruct the step-by-step log
    # that run_react_agent prints during a live run.
    artifact = pinned_run.agent_artifacts()[0]
    p_yes, rationale, traces, tokens = (
        artifact.p_yes,
        artifact.rationale,
        artifact.traces,
        artifact.token_usage,
    )
    if not any(t.action == "forecast" for t in traces):
        p_yes = None
        rationale = "No forecast action in this capture; the saved 0.5 is a legacy sentinel"
    search_name = pinned_run.params.get("search_client", "replay (pinned trace)")
    print(f"Search: {search_name}\n")
    print(f"Question: {question.question}\n")
    for t in traces:
        if t.action == "search":
            print(f'  Step {t.step}: search("{t.query}") → {len(t.results)} results')
        elif t.action == "forecast":
            print(f"  Step {t.step}: forecast → p_yes={p_yes:.2f}")

Search: TavilySearchClient



Question: Will no Fed rate cuts happen in 2026?

  Step 1: search("Fed rate cuts 2026 expectations FOMC forecast") → 5 results
  Step 2: search("Fed rate cuts 2026 current status how many cuts so far June 2026") → 5 results
  Step 3: search("Fed rate cuts 2026 zero cuts hold rates inflation Reuters June 2026") → 5 results
  Step 4: search("Fed rate cut 2026 any cuts made so far current federal funds rate") → 5 results
  Step 5: search("Fed rate cut 2026 probability rest of year outlook inflation hike") → 5 results


In [11]:
print("--- Forecast ---" if p_yes is not None else "--- Abstention ---")
print(f"p(YES) = {p_yes}")
print(f"Rationale: {rationale}")
print(f"Tokens: {tokens.total_tokens:,}")

--- Abstention ---
p(YES) = None
Rationale: No forecast action in this capture; the saved 0.5 is a legacy sentinel
Tokens: 25,696


## 检查执行轨迹

每一步都会捕获为 `AgentTrace`。这对于**可审计性**至关重要：在生产环境中，你需要知道发出了哪些查询、返回了什么结果，以及 LLM 如何得出其预测。

In [12]:
print(f"Total steps: {len(traces)}")
print(f"Token usage: {tokens.total_tokens:,} tokens\n")

for t in traces:
    if t.action == "search":
        print(f'Step {t.step} [SEARCH] query="{t.query}"')
        for r in t.results[:3]:
            print(f"  → {r.title[:70]}")
    elif t.action == "forecast":
        data = json.loads(t.llm_raw) if t.llm_raw else {}
        print(f"Step {t.step} [FORECAST] p_yes={data.get('p_yes', '?')}")
    print()

Total steps: 5
Token usage: 25,696 tokens

Step 1 [SEARCH] query="Fed rate cuts 2026 expectations FOMC forecast"
  → What's The Fed's Next Move? | J.P. Morgan Global Research
  → Fed Outlook 2026: Rate Forecasts and Fixed Income Strategies
  → Fed predictions for 2026: What experts say about the possibility of ..

Step 2 [SEARCH] query="Fed rate cuts 2026 current status how many cuts so far June 2026"
  → Will US Interest Rates Fall More in 2026? Our Latest Forecast
  → The Fed Just Cut Rates Again. Here's What's Ahead for 2026
  → Fed to hold rates this year, cut calls fade as war inflation ... - Reu

Step 3 [SEARCH] query="Fed rate cuts 2026 zero cuts hold rates inflation Reuters June 2026"
  → Fed to hold rates through 2026 amid persistent inflation: Reuters
  → Nomura pivots away from Fed rate cuts in 2026 as inflation risks linge
  → Fed rate cut pushed back to late 2026 on war-related inflation risks: 

Step 4 [SEARCH] query="Fed rate cut 2026 any cuts made so far current federal

## 持久化运行

一次实时运行（live run）依赖三样明天就不存在的东西：一个未决的市场问题、搜索 API 当天返回的文档，以及一个特定的模型版本。将问题、智能体的步骤，以及完整的提示与回复对话写入 `forecast_traces/` 下的一个 JSON 文件，就把这三者都固定下来了。该文件正是 `RUN_LIVE = False` 读取的内容，因此无论每位读者配置了哪个提供商或搜索后端，上面打印出的会话都相同，也正因如此，一次运行可以在数月后重新检查。

In [13]:
if RUN_LIVE:
    artifact = AgentForecastArtifact(
        agent_id="react_agent",
        p_yes=p_yes,
        rationale=rationale,
        traces=traces,
        token_usage=tokens,
        status="accepted" if p_yes is not None else "abstained",
        failure_reason=rationale if p_yes is None else None,
        execution_mode="synthetic" if "mock" in llm.model_name else "live",
        search_queries_made=sum(1 for t in traces if t.action == "search"),
        sources_consulted=sum(len(t.results) for t in traces),
    )
    run = RunTrace.capture(
        notebook="01_react_reasoning",
        provider=provider_name,
        question=question,
        params={
            "max_steps": MAX_STEPS,
            "warmup_response": response,
            "search_client": search_name,
        },
        agents=[artifact],
        llm_calls=tracer.calls,
        notes="ReAct agent on a live prediction-market question.",
    )
    trace_path = run.save()
    print(
        f"Saved {len(run.llm_calls)} model calls ({run.total_tokens():,} tokens) → {trace_path.name}"
    )
else:
    print(
        f"Replayed {len(pinned_run.llm_calls)} model calls "
        f"({pinned_run.total_tokens():,} tokens) from {PINNED_TRACE}"
    )

Replayed 5 model calls (25,696 tokens) from 01_react_reasoning_20260615T191047Z_04eb6e7c603d.json


## 替换后端

`run_react_agent` 中没有任何地方指定提供商。它会在传入的任何对象上调用 `complete_with_usage`，因此任何满足双方法协议的类都可以被替换：这里的 `MockLLMClient`、本地 Ollama 模型，或商业 API。下面的单元格针对 mock 客户端运行同一循环，其回复是预置的，并给它三轮而不是五轮，因为 mock 只搜索一次然后提交。

在实时路径上，`create_llm_client("")` 会选择第一个密钥存在的提供商，而从环境读取 `LLM_PROVIDER` 会覆盖该选择。因此，持续集成运行无需编辑笔记本即可强制使用确定性回复：

```bash
LLM_PROVIDER=mock uv run python 24_autonomous_agents/01_react_reasoning.py
```

该变量仅在 `RUN_LIVE = True` 时生效；在回放路径上根本不会构造客户端。

六月的捕获记录保留了实际搜索和模型消息。它早于新的验收检查；应检查其理由和来源，而不是假定其具有已验证状态。当前循环会拒绝格式错误或不支持的概率，并且仅当保留了证据时才允许一个最终的综合作答轮次。它可以弃权。引用使支持可检查，但并不自动正确。

下面的 mock 示例是一个明确的合成流程测试，而不是模型评估。

In [14]:
mock_llm = MockLLMClient()
mock_search = create_search_client("mock")

p_mock, _, mock_traces, mock_tokens = run_react_agent(mock_llm, mock_search, question, max_steps=3)

print(f"Mock provider: {mock_llm.model_name}")
print(f"Mock steps: {len(mock_traces)}")
print(f"Mock tokens: {mock_tokens.total_tokens:,}")
print(f"Synthetic p(YES): {p_mock}")

Step 0: search -> 2 eligible results
Mock provider: mock-model
Mock steps: 4
Mock tokens: 2,496
Synthetic p(YES): None


## 关键要点

1. **一个接口，任意后端。** 一个只调用 `complete_with_usage`、不调用其他任何东西的 agent，可以在不修改代码的情况下，从免费的本地模型迁移到商业 API。要针对协议编写循环，而不是针对某家厂商的 SDK。
2. **小的动作空间才能让循环可审计。** 只有两个动作时，每一轮要么是一个可以重新发出的查询，要么是一个可以评分的概率。增加动作，就会增加运行出错的方式，而记录无法解释这些错误。
3. **模型输出是不可信输入。** 按声明的 schema 解析它，并把格式错误的回复退回要求修正，而不是让它到达工具。
4. **预算需要一个可区分的耗尽值。** 如果循环在成功和失败时都返回一个概率，调用方就无法区分判断和超时。返回一个调用方必须分支处理的值，并在类型或记录中说明这一点。
5. **记录对话，而不仅仅是答案。** 将提示、回复和工具结果放在一个文件中，才能使一次运行在市场收盘、模型退役之后仍可复现。

**此处所构建内容的已知局限。** 循环把每条消息都保留在上下文窗口中，因此长会话最终会超出窗口，而且没有摘要或淘汰机制。搜索结果不包含发布日期，因此无法证明一次运行不含后见之明。只有一个 agent 和一个样本，所以概率周围没有离散度。而且没有任何东西对预测进行评分：agent 的优劣只取决于其过去概率如何结算的记录，而这需要已揭晓结果的问题和合适的评分规则。

**下一步**：[`02_tool_contracts`](02_tool_contracts.ipynb) 为搜索工具提供类型化契约、发布日期过滤器和来源策略。